In [ ]:
[]

# Model Comparison and Threshold Optimization

This notebook compares candidate models trained on the realistic feature set, excluding `duration`.

We want to answer the practical business question: which model gives the highest usable predictive performance without violating the pre-contact constraint? We also tune the classification threshold because in imbalanced datasets the default threshold of 0.5 may not be the best operational choice.

The comparison focuses on a realistic business-oriented objective: maximize recall and F1 while controlling false positives enough to keep campaign effort efficient.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from src.model_comparison import compare_models, optimize_threshold
from src.data_loader import load_bank_data

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

raw_df = load_bank_data('data/bank_marketing.csv')

models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'decision_tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'random_forest': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced',
        min_samples_leaf=5,
    ),
}

results = compare_models(raw_df, models=models)
print('Model comparison results:')
for name, metrics in results.items():
    print(name, metrics)

# threshold tuning on the logistic regression model
from sklearn.model_selection import train_test_split
from src.model_comparison import prepare_comparison_data
from src.model_comparison import _make_preprocessor
from sklearn.pipeline import Pipeline

X, y = prepare_comparison_data(raw_df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logistic = Pipeline([
    ('preprocessor', _make_preprocessor(X_train)),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
logistic.fit(X_train, y_train)

threshold, best_metrics = optimize_threshold(logistic, X_test, y_test)
print('\nBest threshold by F1:', threshold)
print('Best threshold metrics:', best_metrics)

# plot the threshold sweep
thresholds = np.linspace(0.1, 0.9, 81)
metric_rows = []
probs = logistic.predict_proba(X_test)[:, 1]
for t in thresholds:
    pred = (probs >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score, f1_score
    metric_rows.append({
        'threshold': t,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    })

table = pd.DataFrame(metric_rows)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(table['threshold'], table['precision'], label='precision', marker='o', markersize=3)
ax.plot(table['threshold'], table['recall'], label='recall', marker='o', markersize=3)
ax.plot(table['threshold'], table['f1'], label='f1', marker='o', markersize=3)
ax.axvline(threshold, color='black', linestyle='--', alpha=0.8, label=f'best threshold = {threshold:.2f}')
ax.set_title('Threshold sweep for the realistic bank marketing model')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Metric value')
ax.legend()
plt.show()
